In [1]:
# STEP 1 — IMPORT LIBRARIES
# =========================================================
 
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"    # suppress TF info logs
import cv2
import tensorflow as tf
 
import tensorflow as tf
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing import image
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input
#                                                      ^^^^^^^^^^^^^^
# All 3 models share MobileNetV2 backbone → same preprocess_input for every stage
 
from PIL import Image
 

In [2]:
# STEP 2 — LOAD ALL THREE MODELS
# =========================================================
 
plant_model  = load_model("plant_identifier.keras")   # Stage 1
apple_model  = load_model("Apple_disease.keras")       # Stage 2-A
potato_model = load_model("Potato_disease.keras")      # Stage 2-B
grape_model  = load_model("Grape_disease.keras")       # Stage 2-C
#pepper_model = load_model("Pepper_disease.keras")      # Stage 2-D  (manual override)
 
print("✓ Stage 1 — Plant identifier loaded")
print("  Output classes :", plant_model.output_shape[-1])    # 5
 
print("✓ Stage 2 — Apple  disease model loaded")
print("  Output classes :", apple_model.output_shape[-1])    # 4
 
print("✓ Stage 2 — Potato disease model loaded")
print("  Output classes :", potato_model.output_shape[-1])   # 3
 
print("✓ Stage 2 — Grape  disease model loaded")
print("  Output classes :", grape_model.output_shape[-1])    # 4
 
#print("✓ Stage 2 — Pepper disease model loaded")
#print("  Output classes :", pepper_model.output_shape[-1])   # 2

✓ Stage 1 — Plant identifier loaded
  Output classes : 7
✓ Stage 2 — Apple  disease model loaded
  Output classes : 4
✓ Stage 2 — Potato disease model loaded
  Output classes : 3
✓ Stage 2 — Grape  disease model loaded
  Output classes : 4


In [3]:
# STEP 3 — DEFINE CLASS LABELS
# STAGE 1 : PLANT CLASSES  (5 classes)

plant_classes = [
    "Apple",       # index 0
    "Corn",        # index 1
    "Grape",       # index 2
    "Others",
    "Potato",      # index 3
    "Tomato",      # index 4
    "Pepper"
]
# STAGE 2-A : APPLE DISEASE CLASSES  (4 classes)
# ----------------------------------------------------------
apple_classes = [
    "Apple Scab",
    "Black Rot",
    "Cedar Apple Rust",
    "Healthy",
]
 
# ----------------------------------------------------------
# STAGE 2-B : POTATO DISEASE CLASSES  (3 classes)
# ----------------------------------------------------------
potato_classes = [
    "Early Blight",
    "Late Blight",
    "Healthy",
]
 
# ----------------------------------------------------------
# STAGE 2-C : GRAPE DISEASE CLASSES  (4 classes)
# ----------------------------------------------------------
grape_classes = [
    "Black Rot",
    "Esca (Black Measles)",
    "Leaf Blight (Isariopsis Leaf Spot)",
    "Healthy",
]

# STAGE 2-D : PEPPER DISEASE CLASSES  (2 classes)
# ----------------------------------------------------------
# ⚠  Pepper is NOT in Stage-1's plant list.
#    Use plant_override="Pepper" to bypass Stage 1.
# ----------------------------------------------------------
'''pepper_classes = [
    "Bacterial Spot",
    "Healthy",
]'''


DISEASE_REGISTRY = {
    "Apple":  {"model": apple_model,  "classes": apple_classes},
    "Potato": {"model": potato_model, "classes": potato_classes},
    "Grape":  {"model": grape_model,  "classes": grape_classes},
   # "Pepper": {"model": pepper_model, "classes": pepper_classes},  # manual override
    # Add more as you collect new disease models:
    # "Tomato": {"model": tomato_model, "classes": tomato_classes},
}

In [4]:
# TREATMENT ADVICE
# ----------------------------------------------------------
TREATMENT_ADVICE = {
    # Apple
    "Apple Scab":
        "Apply fungicides (captan/myclobutanil) at bud-break. "
        "Remove infected leaves. Improve air circulation.",
    "Black Rot":
        "Prune infected wood 8–12 inches below cankers. "
        "Apply copper-based fungicide. Avoid overhead irrigation.",
    "Cedar Apple Rust":
        "Remove nearby juniper/cedar hosts if possible. "
        "Apply protective fungicide (mancozeb/myclobutanil) before infection periods.",
    # Potato
    "Early Blight":
        "Apply chlorothalonil or mancozeb fungicide. "
        "Rotate crops; avoid wetting foliage. Remove debris after harvest.",
    "Late Blight":
        "URGENT: Apply metalaxyl or cymoxanil immediately. "
        "Destroy infected plants. Report to local agricultural authority.",
    # Grape
    "Esca (Black Measles)":
        "Remove and destroy infected wood. Apply trunk wound protectants. "
        "Avoid large pruning cuts. No curative treatment exists — prevention is key.",
    "Leaf Blight (Isariopsis Leaf Spot)":
        "Apply copper-based fungicide or mancozeb. "
        "Remove infected leaves. Ensure good canopy air circulation.",
    '''# Pepper
    "Bacterial Spot":
        "Apply copper-based bactericide at first sign. "
        "Avoid overhead irrigation. Use certified disease-free seeds. "
        "Rotate crops for 2–3 years.",'''
    # Shared
    "Healthy":
        "No disease detected. Maintain regular watering and fertilisation schedule.",
}

In [5]:


def preprocess_image(img_path):
    img = image.load_img(img_path, target_size=(224, 224))
    img_array = image.img_to_array(img)

    # ✅ ADD THIS LINE — forces any palette/transparency PNG to RGB
    img_array = np.array(
        Image.fromarray(img_array.astype("uint8")).convert("RGB"),
        dtype=np.float32
    )

    img_array = np.expand_dims(img_array, axis=0)
    img_array = preprocess_input(img_array)
    return img_array

In [6]:
# STEP 4 — IMAGE PREPROCESSING  (shared by all 3 models)
# ======================================================== 


def preprocess_image(img_path, target_size=(224, 224)):
    """
    Improved preprocessing that isolates the dominant leaf
    before feeding to the model.
    Works for both clean lab images and messy real-world photos.
    """
    # ── Load image ────────────────────────────────────────────────────
    img_bgr  = cv2.imread(img_path)
    img_rgb  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    h, w     = img_rgb.shape[:2]

    # ── Step 1: isolate green plant material ──────────────────────────
    # Convert to HSV — green leaves sit in hue range 25–95
    hsv       = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2HSV)
    lower_g   = np.array([25,  40,  40])
    upper_g   = np.array([95, 255, 255])
    mask      = cv2.inRange(hsv, lower_g, upper_g)

    # Clean up small noise blobs
    kernel    = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask      = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    mask      = cv2.morphologyEx(mask, cv2.MORPH_OPEN,  kernel)

    # ── Step 2: find the LARGEST contour (= dominant leaf) ────────────
    contours, _ = cv2.findContours(
        mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE
    )

    if contours:
        # Pick the biggest green blob
        largest   = max(contours, key=cv2.contourArea)
        x, y, bw, bh = cv2.boundingRect(largest)

        # Add 10% padding around the bounding box
        pad_x = int(bw * 0.10)
        pad_y = int(bh * 0.10)
        x1    = max(0, x - pad_x)
        y1    = max(0, y - pad_y)
        x2    = min(w, x + bw + pad_x)
        y2    = min(h, y + bh + pad_y)

        # Crop to the dominant leaf
        img_rgb = img_rgb[y1:y2, x1:x2]
    else:
        # No green region found — fall back to centre crop
        side    = min(h, w)
        cy, cx  = h // 2, w // 2
        img_rgb = img_rgb[cy-side//2 : cy+side//2,
                          cx-side//2 : cx+side//2]

    # ── Step 3: standard resize + normalise ───────────────────────────
    img_pil   = Image.fromarray(img_rgb).resize(target_size)
    img_arr   = np.array(img_pil, dtype=np.float32)
    img_arr   = preprocess_input(img_arr)          # scales to [-1, 1]
    return np.expand_dims(img_arr, axis=0)         # (1, 224, 224, 3)

In [7]:
# STEP 5 — STAGE 1 : PLANT IDENTIFICATION
# =========================================================
 
def identify_plant(img_path, confidence_threshold=40.0):
    """
    Stage 1 — identify the plant species in the image.
 
    Parameters
    ----------
    img_path              : str    Path to the leaf image
    confidence_threshold  : float  Minimum % confidence to accept
                                   the prediction (default 40 %)
 
    Returns
    -------
    plant_name   : str or None   Identified plant; None if below threshold
    confidence   : float         Confidence percentage
    all_scores   : list          [(plant_name, score%), ...] all classes
    """
    img_array  = preprocess_image(img_path)
    raw_preds  = plant_model.predict(img_array, verbose=0)[0]
 
    predicted_index = int(np.argmax(raw_preds))
    confidence      = float(np.max(raw_preds)) * 100
    plant_name      = plant_classes[predicted_index]
 
    all_scores = [
        (plant_classes[i], float(raw_preds[i]) * 100)
        for i in range(len(plant_classes))
    ]
 
    if confidence < confidence_threshold:
        print(f"  ⚠  Low confidence ({confidence:.1f}% < threshold {confidence_threshold}%)")
        print(f"     Best guess: {plant_name} — treat result with caution.")
 
    return plant_name, confidence, all_scores
 

In [8]:
# STEP 6 — STAGE 2 : DISEASE DETECTION
# =========================================================
 
def detect_disease(img_path, plant_name):
    """
    Stage 2 — detect disease for the identified plant.
 
    Parameters
    ----------
    img_path   : str   Path to the leaf image
    plant_name : str   Plant name from Stage 1
 
    Returns
    -------
    disease    : str or None   Predicted disease label
    confidence : float         Confidence percentage
    all_scores : list          [(disease_name, score%), ...]
    advice     : str           Treatment recommendation
    """
    if plant_name not in DISEASE_REGISTRY:
        print(f"  ℹ  No disease model available for '{plant_name}'.")
        print(f"     Supported plants: {list(DISEASE_REGISTRY.keys())}")
        return None, 0, [], "No disease model available for this plant."
 
    entry       = DISEASE_REGISTRY[plant_name]
    model       = entry["model"]
    class_names = entry["classes"]
 
    img_array  = preprocess_image(img_path)
    raw_preds  = model.predict(img_array, verbose=0)[0]
 
    predicted_index = int(np.argmax(raw_preds))
    confidence      = float(np.max(raw_preds)) * 100
    disease         = class_names[predicted_index]
 
    all_scores = [
        (class_names[i], float(raw_preds[i]) * 100)
        for i in range(len(class_names))
    ]
 
    advice = TREATMENT_ADVICE.get(disease, "Consult a local agronomist.")
 
    return disease, confidence, all_scores, advice
 

In [9]:
# STEP 7 — FULL PIPELINE (Stage 1 → Stage 2)
# =========================================================
 
def run_pipeline(img_path, confidence_threshold=40.0, plant_override=None):
    """
    Run the complete two-stage detection pipeline.
 
    Stage 1 : plant_identifier.keras  → plant name
    Stage 2 : disease model           → disease + advice
 
    Parameters
    ----------
    img_path             : str    Path to the leaf image
    confidence_threshold : float  Minimum Stage-1 confidence (default 40 %)
    plant_override       : str or None
        Skip Stage 1 and force a specific plant.
        Required for plants not in Stage-1 (e.g. "Pepper").
        Example: run_pipeline("leaf.jpg", plant_override="Pepper")
 
    Returns
    -------
    dict with keys:
        plant, plant_confidence, plant_scores,
        disease, disease_confidence, disease_scores,
        advice
    """
    print(f"\n{'='*52}")
    print(f"  IMAGE : {img_path}")
    print(f"{'='*52}")
 
    # ── Stage 1 ──────────────────────────────────────────
    if plant_override:
        # Skip Stage 1 entirely — use the provided plant name
        plant    = plant_override.strip().capitalize()
        p_conf   = 100.0
        p_scores = [(plant, 100.0)]
        print(f"\n  [ STAGE 1 ]  Skipped — manual override: {plant}")
    else:
        print("\n  [ STAGE 1 ]  Identifying plant…")
        plant, p_conf, p_scores = identify_plant(img_path, confidence_threshold)
        print(f"  ✓ Plant      : {plant}  ({p_conf:.1f}%)")

        # ✅ NEW — reject if Stage 1 says "Other"
        if plant == 'Other':
            print('  ✗ REJECTED — Stage 1 identified image as non-leaf')
            return {
                'plant':             'Not a plant leaf',
                'plant_confidence':  p_conf,
                'plant_scores':      p_scores,
                'disease':           None,
                'disease_confidence': 0.0,
                'disease_scores':    [],
                'advice':            'Please upload a clear photo of a plant leaf.',
                'screener_rejected': True,
            }
 
    # ── Stage 2 ──────────────────────────────────────────
    print("\n  [ STAGE 2 ]  Detecting disease…")
    disease, d_conf, d_scores, advice = detect_disease(img_path, plant)
 
    if disease:
        print(f"  ✓ Disease    : {disease}  ({d_conf:.1f}%)")
    print(f"{'='*52}\n")
 
    return {
        "plant":              plant,
        "plant_confidence":   p_conf,
        "plant_scores":       p_scores,
        "disease":            disease,
        "disease_confidence": d_conf,
        "disease_scores":     d_scores,
        "advice":             advice,
    }
 

In [10]:
# STEP 8 — DISPLAY FULL RESULT
# =========================================================

def _bar_color(conf):
    if conf >= 80: return "#2ecc71"    # green  — high
    if conf >= 55: return "#f39c12"    # orange — medium
    return "#e74c3c"                   # red    — low


def show_result(img_path, confidence_threshold=40.0, plant_override=None):
    """
    Run the pipeline and display a 3-panel visual report:
      Panel 1 — Leaf image
      Panel 2 — Stage 1 plant confidence bars
      Panel 3 — Stage 2 disease confidence bars
    Plus a printed text summary with treatment advice.

    Parameters
    ----------
    img_path             : str
    confidence_threshold : float  (default 40 %)
    plant_override       : str or None
        Skip Stage 1 (use for Pepper or any plant not in Stage-1).
        Example: show_result("leaf.jpg", plant_override="Pepper")
    """
    result = run_pipeline(img_path, confidence_threshold, plant_override)

    plant   = result["plant"]
    disease = result["disease"]
    p_conf  = result["plant_confidence"]
    d_conf  = result["disease_confidence"]

    img = Image.open(img_path)

    fig = plt.figure(figsize=(16, 5), facecolor="#f8f9fa")
    gs  = gridspec.GridSpec(1, 3, figure=fig, wspace=0.4)

    # ── Panel 1 : Leaf image ──────────────────────────────
    ax0 = fig.add_subplot(gs[0])
    ax0.imshow(img)
    ax0.axis("off")

    status = disease if disease else "No model"
    override_note = "  (manual)" if plant_override else f"  ({p_conf:.1f}%)"
    ax0.set_title(
        f"Plant   : {plant}{override_note}\n"
        f"Disease : {status}"
        + (f"  ({d_conf:.1f}%)" if disease else ""),
        fontsize=11, fontweight="bold", color="#2c3e50", pad=10
    )

    # ── Panel 2 : Stage 1 bars ────────────────────────────
    ax1 = fig.add_subplot(gs[1])
    ax1.set_facecolor("#f8f9fa")

    if plant_override:
        # Stage 1 was skipped — show a simple notice instead of bars
        ax1.text(0.5, 0.5,
                 f"Stage 1 skipped\n(manual override)\nPlant = {plant}",
                 ha="center", va="center", fontsize=11, color="#7f8c8d",
                 transform=ax1.transAxes)
    else:
        p_labels = [s[0] for s in result["plant_scores"]]
        p_vals   = [s[1] for s in result["plant_scores"]]
        p_colors = [_bar_color(p_conf) if s[0] == plant else "#bdc3c7"
                    for s in result["plant_scores"]]

        yp = range(len(p_labels))
        bars1 = ax1.barh(list(yp), p_vals, color=p_colors,
                         edgecolor="white", height=0.5)
        for bar, val in zip(bars1, p_vals):
            ax1.text(min(val + 1.5, 97), bar.get_y() + bar.get_height() / 2,
                     f"{val:.1f}%", va="center", fontsize=9, color="#2c3e50")
        ax1.set_yticks(list(yp))
        ax1.set_yticklabels(p_labels, fontsize=10)
        ax1.set_xlim(0, 110)
        ax1.set_xlabel("Confidence (%)", fontsize=9)

    ax1.set_title("Stage 1 — Plant ID", fontsize=11,
                  fontweight="bold", color="#2c3e50")
    ax1.spines[["top", "right"]].set_visible(False)

    # ── Panel 3 : Stage 2 bars ────────────────────────────
    ax2 = fig.add_subplot(gs[2])
    ax2.set_facecolor("#f8f9fa")

    if disease and result["disease_scores"]:
        d_labels = [s[0] for s in result["disease_scores"]]
        d_vals   = [s[1] for s in result["disease_scores"]]
        d_colors = [_bar_color(d_conf) if s[0] == disease else "#bdc3c7"
                    for s in result["disease_scores"]]

        yd = range(len(d_labels))
        bars2 = ax2.barh(list(yd), d_vals, color=d_colors,
                         edgecolor="white", height=0.5)
        for bar, val in zip(bars2, d_vals):
            ax2.text(min(val + 1.5, 97), bar.get_y() + bar.get_height() / 2,
                     f"{val:.1f}%", va="center", fontsize=9, color="#2c3e50")
        ax2.set_yticks(list(yd))
        ax2.set_yticklabels(d_labels, fontsize=10)
        ax2.set_xlim(0, 110)
        ax2.set_xlabel("Confidence (%)", fontsize=9)
    else:
        ax2.text(0.5, 0.5, "No disease model\nfor this plant",
                 ha="center", va="center", fontsize=11, color="#95a5a6",
                 transform=ax2.transAxes)

    ax2.set_title(f"Stage 2 — Disease ({plant})", fontsize=11,
                  fontweight="bold", color="#2c3e50")
    ax2.spines[["top", "right"]].set_visible(False)

    plt.suptitle("Two-Stage Plant Disease Detection",
                 fontsize=13, fontweight="bold", color="#2c3e50", y=1.02)
    plt.tight_layout()
    plt.show()

    # ── Printed summary ───────────────────────────────────
    line  = "=" * 55
    hline = "-" * 55

    print(line)
    print("  TWO-STAGE DETECTION REPORT")
    print(line)

    # Stage 1
    print("  STAGE 1 — PLANT IDENTIFICATION")
    pb = int(p_conf / 2)
    print(f"    Plant      : {plant}")
    print(f"    Confidence : {p_conf:.2f}%  [{'█'*pb}{'░'*(50-pb)}]")
    print("    All scores :")
    for cls, sc in sorted(result["plant_scores"], key=lambda x: x[1], reverse=True):
        marker = " ◀ IDENTIFIED" if cls == plant else ""
        print(f"      {cls:<12} {sc:6.2f}%{marker}")

    print(hline)

    # Stage 2
    print("  STAGE 2 — DISEASE DETECTION")
    if disease:
        db = int(d_conf / 2)
        print(f"    Disease    : {disease}")
        print(f"    Confidence : {d_conf:.2f}%  [{'█'*db}{'░'*(50-db)}]")
        print("    All scores :")
        for cls, sc in sorted(result["disease_scores"], key=lambda x: x[1], reverse=True):
            marker = " ◀ DETECTED" if cls == disease else ""
            print(f"      {cls:<25} {sc:6.2f}%{marker}")
        print(hline)
        print("  ADVICE :")
        # word-wrap at 50 chars
        words, line_buf, wrapped = result["advice"].split(), "", []
        for w in words:
            if len(line_buf) + len(w) + 1 > 50:
                wrapped.append(line_buf); line_buf = w
            else:
                line_buf = (line_buf + " " + w).strip()
        if line_buf: wrapped.append(line_buf)
        for ln in wrapped:
            print(f"    {ln}")
    else:
        print(f"    {result['advice']}")

    print("=" * 55)



In [11]:
# STEP 9 — BATCH PREDICTION  (optional)
# =========================================================

def predict_batch(img_paths, confidence_threshold=40.0, plant_overrides=None):
    """
    Run the full pipeline on a list of images and return
    a summary table.

    Parameters
    ----------
    img_paths        : list of str   Image file paths
    confidence_threshold : float
    plant_overrides  : list of str or None
        Optional per-image plant override (same length as img_paths).
        Pass None for images that should go through Stage 1 normally.
        Example:
            predict_batch(
                ["grape.jpg", "pepper.jpg", "apple.jpg"],
                plant_overrides=[None, "Pepper", None]
            )

    Returns
    -------
    list of result dicts
    """
    results = []
    for i, path in enumerate(img_paths):
        override = plant_overrides[i] if plant_overrides else None
        print(f"\n── Image {i+1}/{len(img_paths)} ──")
        result = run_pipeline(path, confidence_threshold, plant_override=override)
        results.append({"file": path, **result})

    # Print summary table
    print("\n" + "=" * 75)
    print(f"  {'FILE':<20} {'PLANT':<12} {'P.CONF':>7}  {'DISEASE':<25} {'D.CONF':>7}")
    print("=" * 75)
    for r in results:
        d_label  = r["disease"] if r["disease"] else "—"
        d_conf   = f"{r['disease_confidence']:.1f}%" if r["disease"] else "  —"
        p_c_str  = "manual" if r["plant_confidence"] == 100.0 and len(r["plant_scores"]) == 1 \
                   else f"{r['plant_confidence']:.1f}%"
        print(f"  {r['file']:<20} {r['plant']:<12} {p_c_str:>7}"
              f"  {d_label:<25} {d_conf:>7}")
    print("=" * 75)

    return results


In [12]:
'''# STEP 5-A — GRAD-CAM CORE FUNCTIONS (Keras 3 Compatible Fix)
# ============================================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from PIL import Image

# ── Helper: find MobileNetV2 inside any model ──────────────────────────────
def _find_mobilenet_base(model):
    for layer in model.layers:
        if isinstance(layer, tf.keras.Model) and layer.name.startswith("mobilenetv2"):
            return layer
    raise ValueError(
        f"No MobileNetV2 sub-model found.\n"
        f"Layer names: {[l.name for l in model.layers]}"
    )


# ── Core Grad-CAM — Keras 3 compatible ────────────────────────────────────
def get_gradcam_heatmap(model, img_array, pred_index=None,
                        last_conv_layer_name="out_relu"):
    """
    Compute Grad-CAM heatmap. Works with Sequential AND Functional models
    under Keras 3 (Python 3.13 / TF 2.16+).

    Parameters
    ----------
    model                : keras.Model   Any of the five project models.
    img_array            : np.ndarray    Preprocessed image (1, 224, 224, 3).
    pred_index           : int or None   Class to explain. None = argmax.
    last_conv_layer_name : str           Last conv layer in MobileNetV2.
                                         'out_relu' works for all models.
    Returns
    -------
    heatmap    : np.ndarray  shape (7, 7), float32 in [0, 1]
    pred_index : int
    """
    base_model = _find_mobilenet_base(model)

    # ── Step 1: build a sub-model from MobileNetV2 input → conv layer ──
    # MobileNetV2 is always Functional → this ALWAYS works in Keras 3
    conv_layer = base_model.get_layer(last_conv_layer_name)
    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=conv_layer.output          # (1, 7, 7, 1280)
    )

    # ── Step 2: find which index base_model sits at in the outer model ──
    base_idx = next(
        i for i, lyr in enumerate(model.layers) if lyr is base_model
    )

    # Layers that come AFTER MobileNetV2 (GAP, Dense, Dropout, Softmax…)
    remaining_layers = model.layers[base_idx + 1:]

    img_tensor = tf.cast(img_array, tf.float32)

    # ── Step 3: GradientTape — watch conv_outputs explicitly ────────────
    with tf.GradientTape() as tape:
        # Extract 7×7×1280 feature maps
        conv_outputs = feature_extractor(img_tensor, training=False)

        # Tell the tape to watch this intermediate tensor
        tape.watch(conv_outputs)

        # Manually thread conv_outputs through the remaining layers
        # so that gradients flow: class_score → conv_outputs ✓
        x = conv_outputs
        for layer in remaining_layers:
            x = layer(x, training=False)
        predictions = x                    # (1, num_classes)

        if pred_index is None:
            pred_index = int(tf.argmax(predictions[0]))

        class_score = predictions[:, pred_index]

    # ── Step 4: compute Grad-CAM weights ────────────────────────────────
    grads = tape.gradient(class_score, conv_outputs)  # (1, 7, 7, 1280)

    if grads is None:
        raise RuntimeError(
            "Gradient is None — the conv layer output is not connected "
            "to the class score. Check last_conv_layer_name."
        )

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))  # (1280,)

    conv_map   = conv_outputs[0]                            # (7, 7, 1280)
    heatmap    = conv_map @ pooled_grads[..., tf.newaxis]   # (7, 7, 1)
    heatmap    = tf.squeeze(heatmap)                        # (7, 7)

    # ReLU + normalise to [0, 1]
    heatmap = tf.maximum(heatmap, 0)
    max_val = tf.math.reduce_max(heatmap)
    if max_val > 0:
        heatmap = heatmap / max_val

    return heatmap.numpy(), pred_index


# ── Overlay heatmap on original image ─────────────────────────────────────
def overlay_gradcam(img_path, heatmap, alpha=0.45, colormap=cm.jet):
    """
    Superimpose Grad-CAM heatmap on the original leaf image.

    Returns
    -------
    overlay : np.ndarray  uint8 RGB (224, 224, 3)
    """
    orig_img = np.array(Image.open(img_path).convert("RGB").resize((224, 224)))

    # Upsample 7×7 → 224×224 via bilinear interpolation
    heatmap_pil = Image.fromarray(np.uint8(255 * heatmap)) \
                       .resize((224, 224), resample=Image.BILINEAR)
    heatmap_up  = np.array(heatmap_pil) / 255.0

    # Apply colourmap (drop alpha channel)
    colored  = colormap(heatmap_up)[:, :, :3]
    colored  = np.uint8(colored * 255)

    overlay  = np.uint8(colored * alpha + orig_img * (1 - alpha))
    return overlay


print("✓ Grad-CAM functions loaded (Keras 3 compatible)")
'''

✓ Grad-CAM functions loaded (Keras 3 compatible)


In [15]:
# =========================================================
# FULL GRAD-CAM INTEGRATION
# FOR YOUR TWO-STAGE PLANT DISEASE SYSTEM
# =========================================================

# =========================================================
# IMPORTS
# =========================================================

import tensorflow as tf
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.cm as cm

from PIL import Image
from tensorflow.keras.preprocessing import image


# =========================================================
# FIND MOBILENETV2 BASE MODEL
# =========================================================

def find_mobilenet_base(model):

    for layer in model.layers:

        if isinstance(layer, tf.keras.Model):

            if "mobilenetv2" in layer.name.lower():
                return layer

    raise ValueError("MobileNetV2 backbone not found.")


# =========================================================
# GENERATE GRAD-CAM HEATMAP
# =========================================================

def get_gradcam_heatmap(
        model,
        img_array,
        pred_index=None,
        last_conv_layer_name="out_relu"
):

    # -----------------------------------------------------
    # FIND MOBILENETV2 BACKBONE
    # -----------------------------------------------------

    base_model = find_mobilenet_base(model)

    # -----------------------------------------------------
    # GET LAST CONVOLUTION LAYER
    # -----------------------------------------------------

    conv_layer = base_model.get_layer(last_conv_layer_name)

    # -----------------------------------------------------
    # CREATE FEATURE EXTRACTOR MODEL
    # -----------------------------------------------------

    feature_extractor = tf.keras.Model(
        inputs=base_model.input,
        outputs=conv_layer.output
    )

    # -----------------------------------------------------
    # FIND REMAINING LAYERS AFTER MOBILENETV2
    # -----------------------------------------------------

    base_idx = next(
        i for i, lyr in enumerate(model.layers)
        if lyr is base_model
    )

    remaining_layers = model.layers[base_idx + 1:]

    img_tensor = tf.cast(img_array, tf.float32)

    # -----------------------------------------------------
    # GRADIENT TAPE
    # -----------------------------------------------------

    with tf.GradientTape() as tape:

        # Feature maps
        conv_outputs = feature_extractor(img_tensor, training=False)

        tape.watch(conv_outputs)

        # Manual forward pass
        x = conv_outputs

        for layer in remaining_layers:
            x = layer(x, training=False)

        predictions = x

        # Predicted class
        if pred_index is None:
            pred_index = int(tf.argmax(predictions[0]))

        class_score = predictions[:, pred_index]

    # -----------------------------------------------------
    # COMPUTE GRADIENTS
    # -----------------------------------------------------

    grads = tape.gradient(class_score, conv_outputs)

    if grads is None:
        raise RuntimeError("Gradients are None.")

    # -----------------------------------------------------
    # GLOBAL AVERAGE POOLING
    # -----------------------------------------------------

    pooled_grads = tf.reduce_mean(grads, axis=(0, 1, 2))

    # -----------------------------------------------------
    # FEATURE MAPS
    # -----------------------------------------------------

    conv_map = conv_outputs[0]

    # -----------------------------------------------------
    # WEIGHTED FEATURE MAP SUM
    # -----------------------------------------------------

    heatmap = tf.reduce_sum(
        conv_map * pooled_grads,
        axis=-1
    )

    # -----------------------------------------------------
    # RELU
    # -----------------------------------------------------

    heatmap = tf.maximum(heatmap, 0)

    # -----------------------------------------------------
    # NORMALIZE
    # -----------------------------------------------------

    max_val = tf.math.reduce_max(heatmap)

    if max_val > 0:
        heatmap = heatmap / max_val

    return heatmap.numpy(), pred_index


# =========================================================
# OVERLAY HEATMAP ON ORIGINAL IMAGE
# =========================================================

def overlay_gradcam(
        img_path,
        heatmap,
        alpha=0.45,
        colormap=cm.jet
):

    # -----------------------------------------------------
    # LOAD ORIGINAL IMAGE
    # -----------------------------------------------------

    orig_img = Image.open(img_path).convert("RGB")
    orig_img = orig_img.resize((224, 224))

    orig_img = np.array(orig_img)

    # -----------------------------------------------------
    # RESIZE HEATMAP
    # -----------------------------------------------------

    heatmap_img = Image.fromarray(
        np.uint8(255 * heatmap)
    )

    heatmap_img = heatmap_img.resize(
        (224, 224),
        resample=Image.BILINEAR
    )

    heatmap_array = np.array(heatmap_img) / 255.0

    # -----------------------------------------------------
    # APPLY COLORMAP
    # -----------------------------------------------------

    colored_heatmap = colormap(heatmap_array)

    # Remove alpha channel
    colored_heatmap = colored_heatmap[:, :, :3]

    colored_heatmap = np.uint8(colored_heatmap * 255)

    # -----------------------------------------------------
    # BLEND
    # -----------------------------------------------------

    overlay = np.uint8(
        colored_heatmap * alpha +
        orig_img * (1 - alpha)
    )

    return overlay


# =========================================================
# COMPLETE GRAD-CAM VISUALIZATION
# =========================================================

def show_gradcam(
        img_path,
        model,
        preprocess_function,
        class_names,
        title="Grad-CAM"
):

    # -----------------------------------------------------
    # PREPROCESS IMAGE
    # -----------------------------------------------------

    img = image.load_img(
        img_path,
        target_size=(224, 224)
    )

    img_array = image.img_to_array(img)

    img_array = np.expand_dims(img_array, axis=0)

    img_array = preprocess_function(img_array)

    # -----------------------------------------------------
    # MODEL PREDICTION
    # -----------------------------------------------------

    predictions = model.predict(img_array, verbose=0)

    pred_index = int(np.argmax(predictions[0]))

    confidence = float(np.max(predictions[0])) * 100

    predicted_class = class_names[pred_index]

    # -----------------------------------------------------
    # GENERATE HEATMAP
    # -----------------------------------------------------

    heatmap, _ = get_gradcam_heatmap(
        model,
        img_array,
        pred_index=pred_index
    )

    # -----------------------------------------------------
    # CREATE OVERLAY
    # -----------------------------------------------------

    overlay = overlay_gradcam(
        img_path,
        heatmap
    )

    # -----------------------------------------------------
    # DISPLAY RESULTS
    # -----------------------------------------------------

    fig, ax = plt.subplots(1, 3, figsize=(16, 5))

    # Original image
    ax[0].imshow(Image.open(img_path))
    ax[0].set_title("Original Image")
    ax[0].axis("off")

    # Heatmap
    ax[1].imshow(heatmap, cmap="jet")
    ax[1].set_title("Grad-CAM Heatmap")
    ax[1].axis("off")

    # Overlay
    ax[2].imshow(overlay)
    ax[2].set_title(
        f"{predicted_class}\nConfidence: {confidence:.2f}%"
    )
    ax[2].axis("off")

    plt.suptitle(title, fontsize=16)

    plt.tight_layout()

    plt.show()

    # -----------------------------------------------------
    # PRINT RESULT
    # -----------------------------------------------------

    print("\n======================================")
    print("GRAD-CAM RESULT")
    print("======================================")

    print(f"Predicted Class : {predicted_class}")
    print(f"Confidence      : {confidence:.2f}%")

    print("======================================")

In [16]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  COMBINED CELL — Upload Button + show_result + show_gradcam             ║
# ║  Replaces both your old show_result() and show_gradcam() run cells.     ║
# ║  Drop this in AFTER all your existing setup cells (Steps 1–5-B).        ║
# ╚══════════════════════════════════════════════════════════════════════════╝

import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.cm as cm
import numpy as np
from PIL import Image
import io, warnings
warnings.filterwarnings("ignore")

TEMP_IMG_PATH = "_uploaded_leaf.jpg"   # temp file for the uploaded image

# ─────────────────────────────────────────────────────────────────────────────
# CORE COMBINED DISPLAY FUNCTION
# ─────────────────────────────────────────────────────────────────────────────
def show_combined(img_path, confidence_threshold=40.0,
                  plant_override=None, alpha=0.45, colormap=cm.jet):
    """
    Unified function that merges show_result + show_gradcam into one figure.

    Layout (2 rows × 4 columns):
    ┌──────────┬──────────────┬──────────────┬──────────────┐
    │ Original │ S1 Grad-CAM  │ S2 Grad-CAM  │  Pure Heatmap│  ← Row 0
    ├──────────┴──────────────┼──────────────┼──────────────┤
    │  Stage-1 Confidence     │ Stage-2 Conf │  Treatment   │  ← Row 1
    └─────────────────────────┴──────────────┴──────────────┘
    """
    # ── 1. Run pipeline ───────────────────────────────────────────────────
    result      = run_pipeline(img_path, confidence_threshold, plant_override)
    plant       = result["plant"]
    disease     = result["disease"]
    p_conf      = result["plant_confidence"]
    d_conf      = result["disease_confidence"]
    advice      = result["advice"]

    img_array   = preprocess_image(img_path)
    orig_img    = np.array(Image.open(img_path).convert("RGB"))

    # ── 2. Stage-1 Grad-CAM ───────────────────────────────────────────────
    heatmap_plant = overlay_plant = None
    if not plant_override and plant in plant_classes:
        try:
            heatmap_plant, _ = get_gradcam_heatmap(
                plant_model, img_array,
                pred_index=plant_classes.index(plant)
            )
            overlay_plant = overlay_gradcam(img_path, heatmap_plant, alpha, colormap)
        except Exception as e:
            print(f"  ⚠ Stage-1 Grad-CAM skipped: {e}")

    # ── 3. Stage-2 Grad-CAM ───────────────────────────────────────────────
    heatmap_disease = overlay_disease = None
    if disease and plant in DISEASE_REGISTRY:
        entry = DISEASE_REGISTRY[plant]
        try:
            heatmap_disease, _ = get_gradcam_heatmap(
                entry["model"], img_array,
                pred_index=entry["classes"].index(disease)
            )
            overlay_disease = overlay_gradcam(img_path, heatmap_disease, alpha, colormap)
        except Exception as e:
            print(f"  ⚠ Stage-2 Grad-CAM skipped: {e}")

    # ── 4. All-class probabilities for confidence bars ────────────────────
    s1_probs  = plant_model.predict(img_array, verbose=0)[0]
    s1_labels = plant_classes

    s2_probs = s2_labels = None
    if plant in DISEASE_REGISTRY:
        entry    = DISEASE_REGISTRY[plant]
        s2_probs  = entry["model"].predict(img_array, verbose=0)[0]
        s2_labels = entry["classes"]

    # ── 5. Build figure ───────────────────────────────────────────────────
    BG   = "#12121E"   # canvas
    CARD = "#1E1E30"   # panel background
    TC   = "#E8E8F0"   # text
    ACC  = "#1DB954"   # green accent (healthy)
    RED  = "#E74C3C"   # red accent  (diseased)
    BLUE = "#4A9EFF"   # bar colour
    GOLD = "#F0A500"   # selected bar

    is_healthy = disease and "healthy" in disease.lower()
    disease_colour = ACC if is_healthy else RED

    fig = plt.figure(figsize=(22, 12), facecolor=BG)
    gs  = gridspec.GridSpec(
        2, 4, figure=fig,
        hspace=0.38, wspace=0.28,
        height_ratios=[1.15, 0.85]
    )

    def _ax(row, col, colspan=1):
        ax = fig.add_subplot(gs[row, col] if colspan == 1
                             else gs[row, col:col+colspan])
        ax.set_facecolor(CARD)
        for s in ax.spines.values():
            s.set_edgecolor("#2E2E45")
            s.set_linewidth(0.6)
        return ax

    # ── Row 0: image panels ───────────────────────────────────────────────
    # Panel 0 — Original
    ax0 = _ax(0, 0)
    ax0.imshow(orig_img)
    ax0.axis("off")
    ax0.set_title("Original Leaf", color=TC, fontsize=11,
                  fontweight="bold", pad=9)

    # Panel 1 — Stage-1 Grad-CAM
    ax1 = _ax(0, 1)
    if overlay_plant is not None:
        ax1.imshow(overlay_plant)
        ax1.set_title(f"Stage 1 — Grad-CAM\n{plant}  {p_conf:.1f}%",
                      color=ACC, fontsize=10, fontweight="bold", pad=9)
    else:
        ax1.text(0.5, 0.5,
                 f"Stage 1 skipped\n(override: {plant})",
                 ha="center", va="center", color="#6E6E8E",
                 fontsize=10, transform=ax1.transAxes)
        ax1.set_title("Stage 1 — Grad-CAM", color=TC,
                      fontsize=11, fontweight="bold", pad=9)
    ax1.axis("off")

    # Panel 2 — Stage-2 Grad-CAM
    ax2 = _ax(0, 2)
    if overlay_disease is not None:
        ax2.imshow(overlay_disease)
        ax2.set_title(
            f"Stage 2 — Grad-CAM\n{disease}  {d_conf:.1f}%",
            color=disease_colour, fontsize=10, fontweight="bold", pad=9
        )
    else:
        ax2.text(0.5, 0.5, "No disease model\nfor this plant",
                 ha="center", va="center", color="#6E6E8E",
                 fontsize=10, transform=ax2.transAxes)
        ax2.set_title("Stage 2 — Grad-CAM", color=TC,
                      fontsize=11, fontweight="bold", pad=9)
    ax2.axis("off")

    # Panel 3 — Pure activation heatmap
    ax3 = _ax(0, 3)
    hm = heatmap_disease if heatmap_disease is not None else heatmap_plant
    if hm is not None:
        hm_up = np.array(
            Image.fromarray(np.uint8(255 * hm))
                  .resize((224, 224), resample=Image.BILINEAR)
        )
        im   = ax3.imshow(hm_up, cmap=colormap)
        cbar = plt.colorbar(im, ax=ax3, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(colors=TC, labelsize=7)
        cbar.outline.set_edgecolor("#2E2E45")
        src  = "Disease" if heatmap_disease is not None else "Plant"
        ax3.set_title(f"{src} Activation Map", color=TC,
                      fontsize=11, fontweight="bold", pad=9)
    else:
        ax3.text(0.5, 0.5, "No heatmap\navailable",
                 ha="center", va="center", color="#6E6E8E",
                 fontsize=10, transform=ax3.transAxes)
        ax3.set_title("Activation Map", color=TC,
                      fontsize=11, fontweight="bold", pad=9)
    ax3.axis("off")

    # ── Row 1: confidence charts ──────────────────────────────────────────
    # Stage-1 confidence bars (spans 2 columns for readability)
    ax4 = _ax(1, 0, colspan=2)
    bar_cols_s1 = [ACC if l == plant else BLUE for l in s1_labels]
    bars4 = ax4.barh(s1_labels, s1_probs * 100,
                     color=bar_cols_s1, edgecolor="none", height=0.55)
    ax4.set_xlim(0, 115)
    ax4.set_xlabel("Confidence (%)", color="#9090A8", fontsize=8)
    ax4.tick_params(axis="y", colors=TC, labelsize=9)
    ax4.tick_params(axis="x", colors="#6E6E8E", labelsize=7)
    ax4.set_title("Stage 1 — Plant Identification", color=TC,
                  fontsize=11, fontweight="bold", pad=9)
    for bar, prob in zip(bars4, s1_probs):
        ax4.text(bar.get_width() + 1.5,
                 bar.get_y() + bar.get_height() / 2,
                 f"{prob*100:.1f}%", va="center",
                 color=TC, fontsize=9, fontweight="bold")
    for sp in ["top", "right"]:
        ax4.spines[sp].set_visible(False)

    # Stage-2 confidence bars
    ax5 = _ax(1, 2)
    if s2_probs is not None and s2_labels is not None:
        bar_cols_s2 = [disease_colour if l == disease else GOLD
                       for l in s2_labels]
        bars5 = ax5.barh(s2_labels, s2_probs * 100,
                         color=bar_cols_s2, edgecolor="none", height=0.55)
        ax5.set_xlim(0, 120)
        ax5.set_xlabel("Confidence (%)", color="#9090A8", fontsize=8)
        ax5.tick_params(axis="y", colors=TC, labelsize=9)
        ax5.tick_params(axis="x", colors="#6E6E8E", labelsize=7)
        for bar, prob in zip(bars5, s2_probs):
            ax5.text(bar.get_width() + 1.5,
                     bar.get_y() + bar.get_height() / 2,
                     f"{prob*100:.1f}%", va="center",
                     color=TC, fontsize=9, fontweight="bold")
        for sp in ["top", "right"]:
            ax5.spines[sp].set_visible(False)
    else:
        ax5.text(0.5, 0.5, "No disease model\nfor this plant",
                 ha="center", va="center", color="#6E6E8E",
                 fontsize=10, transform=ax5.transAxes)
    ax5.set_title(f"Stage 2 — {plant} Diseases", color=TC,
                  fontsize=11, fontweight="bold", pad=9)

    # Treatment advice panel
    ax6 = _ax(1, 3)
    ax6.axis("off")
    ax6.set_title("Treatment Advice", color=TC,
                  fontsize=11, fontweight="bold", pad=9)

    # word-wrap advice text at ~28 chars
    words, lines, cur = advice.split(), [], ""
    for w in words:
        if len(cur) + len(w) + 1 > 28:
            lines.append(cur); cur = w
        else:
            cur = (cur + " " + w).strip()
    if cur:
        lines.append(cur)

    ax6.text(0.5, 0.72,
             disease if disease else "N/A",
             ha="center", va="center", color=disease_colour,
             fontsize=12, fontweight="bold",
             transform=ax6.transAxes)
    ax6.text(0.5, 0.35,
             "\n".join(lines),
             ha="center", va="center", color="#C8C8DC",
             fontsize=8.5, linespacing=1.55,
             transform=ax6.transAxes,
             bbox=dict(boxstyle="round,pad=0.55",
                       facecolor="#252540",
                       edgecolor="#3A3A58",
                       linewidth=0.8))

    # ── Supertitle ────────────────────────────────────────────────────────
    fig.suptitle(
        f"Plant: {plant}  ({p_conf:.1f}%)          "
        f"Disease: {disease if disease else 'N/A'}"
        + (f"  ({d_conf:.1f}%)" if disease else ""),
        color="#F0F0FF", fontsize=14, fontweight="bold", y=1.01
    )

    plt.savefig("gradcam_result.png", dpi=150,
               bbox_inches="tight", facecolor=BG)
    plt.show()

    # ── Text summary ──────────────────────────────────────────────────────
    print("=" * 58)
    print(f"  Plant   : {plant:<20} Confidence: {p_conf:.1f}%")
    print(f"  Disease : {(disease if disease else 'N/A'):<20}"
          + (f" Confidence: {d_conf:.1f}%" if disease else ""))
    print(f"  Advice  : {advice}")
    #print(f"  Saved   : gradcam_result.png")
    print("=" * 58)


# ─────────────────────────────────────────────────────────────────────────────
# UPLOAD WIDGET UI
# ─────────────────────────────────────────────────────────────────────────────
_upload_btn = widgets.FileUpload(
    accept="image/*",
    multiple=False,
    description="📂 Upload Leaf",
    layout=widgets.Layout(width="175px", height="36px"),
)

_conf_slider = widgets.FloatSlider(
    value=40.0, min=0.0, max=100.0, step=5.0,
    description="Min Conf %:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="340px"),
)

_plant_drop = widgets.Dropdown(
    options=["Auto-detect",
             "Apple", "Corn", "Grape",
             "Potato", "Tomato", "Pepper"],
    value="Auto-detect",
    description="Plant Override:",
    style={"description_width": "110px"},
    layout=widgets.Layout(width="280px"),
)

_run_btn = widgets.Button(
    description="▶  Analyse",
    button_style="success",
    layout=widgets.Layout(width="120px", height="36px"),
    style={"font_weight": "bold"},
)

_status = widgets.HTML(
    value="<span style='color:#9090A8;font-size:13px'>"
          "⬆ Upload a leaf image, then click ▶ Analyse.</span>"
)

_out = widgets.Output()


def _on_upload(change):
    if _upload_btn.value:
        _status.value = (
            "<span style='color:#1DB954;font-size:13px'>"
            "✅ Image ready — click ▶ Analyse.</span>"
        )

def _on_run(b):
    _run_btn.disabled = True
    _status.value = ("<span style='color:#F0A500;font-size:13px'>"
                     "⏳ Running pipeline…</span>")
    with _out:
        clear_output(wait=True)
        # ── get uploaded bytes (handles ipywidgets 7 and 8) ──────────────
        try:
            val = _upload_btn.value
            if not val:
                print("⚠  Please upload an image first.")
                _run_btn.disabled = False
                return
            # ipywidgets 8 — val is a tuple of dicts
            if isinstance(val, (list, tuple)):
                content = bytes(val[0]["content"])
            else:
                # ipywidgets 7 — val is a dict keyed by filename
                content = list(val.values())[0]["content"]

            with open(TEMP_IMG_PATH, "wb") as f:
                f.write(content)

            override = _plant_drop.value
            override = None if override == "Auto-detect" else override

            show_combined(
                TEMP_IMG_PATH,
                confidence_threshold=_conf_slider.value,
                plant_override=override,
            )
            _status.value = ("<span style='color:#1DB954;font-size:13px'>"
                             "✅ Done! ")
        except Exception as e:
            print(f"❌ Error: {e}")
            _status.value = (f"<span style='color:#E74C3C;font-size:13px'>"
                             f"❌ Error: {e}</span>")
        finally:
            _run_btn.disabled = False


_upload_btn.observe(_on_upload, names="value")
_run_btn.on_click(_on_run)

# ── Render UI ─────────────────────────────────────────────────────────────
display(
    widgets.VBox([
        widgets.HTML(
            "<div style='font-family:monospace;font-size:15px;"
            "color:#1DB954;font-weight:bold;margin-bottom:8px'>"
            "🌿 Plant Disease Detector</div>"
        ),
        widgets.HBox(
            [_upload_btn, _run_btn],
            layout=widgets.Layout(gap="10px", align_items="center")
        ),
        widgets.HBox(
            [_conf_slider, _plant_drop],
            layout=widgets.Layout(gap="20px", margin_top="6px")
        ),
        _status,
    ],
    layout=widgets.Layout(
        padding="16px",
        border="1px solid #2E2E45",
        border_radius="10px",
        width="680px",
        background_color="#12121E",
    )),
    _out,
)

Output()